# pinn_dispersion.ipynb — Nonlinear Dispersion Curve via PINN Simulation

Uses the segment-by-segment PINN solver (`pinn_ndof_chain.py`) to simulate a **periodic ring**
of N meta-impactor unit cells, then extracts the dispersion relation ω(k) via a 2-D FFT.

**Why this is the "real" dispersion curve:**  
Unlike `dispersion_curve.py` which uses RK4 time integration, here the between-impact dynamics
are solved by a physics-informed neural network (PINN) that satisfies the ODE residual exactly
at collocation points, and impact times are found by root-finding on the frozen network.

**Pipeline:**
1. Periodic ring K matrix (cyclic tridiagonal)  
2. Broadband random ICs (excites all wavenumbers simultaneously)  
3. Two-phase PINN per segment → stitched spatiotemporal array x_n(t)  
4. Resample to uniform time grid  
5. 2-D FFT → S(k, ω) heatmap + ridge extraction

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from scipy.interpolate import interp1d
import os

from pinn_ndof_chain import PIPNNs, find_impact_times, propagate_ics
from dispersion_curve import dispersion_from_2dfft, plot_dispersion_heatmap, \
                             linear_dispersion, extract_ridge

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
np.random.seed(1234)
tf.compat.v1.set_random_seed(1234)
print('TF version:', tf.__version__)

## Physical parameters

Same values as `pinn_ndof_chain_sim.ipynb`.  
The stiffness matrix is **cyclic tridiagonal** (periodic ring), which is required for a well-defined dispersion relation.

In [ ]:
N    = 10       # number of unit cells (= n_dof)
mx   = 1.0      # primary mass
my   = 0.1      # impactor mass
k    = 100.0    # inter-cell spring stiffness
c    = 0.5      # damping coefficient
D    = 1.0      # impact gap
r    = 0.7      # coefficient of restitution

# ── Periodic (ring) stiffness matrix ──────────────────────────────────────
# K_ring[i,i] = 2k  for all i  (not k for the end cells)
# K_ring[i,(i±1)%N] = -k       (wraps around)
K_ring = np.zeros((N, N))
for i in range(N):
    K_ring[i, i]           = 2.0 * k
    K_ring[i, (i - 1) % N] = -k
    K_ring[i, (i + 1) % N] = -k

M_mat = np.diag(mx * np.ones(N))
C_mat = (c / k) * K_ring          # proportional damping

# No external forcing — broadband ICs excite all modes
phi1 = np.array([[0.0]])
phi2 = np.array([[1.0]])           # unused when phi1=0

print('K_ring (diag)  :', np.diag(K_ring))
print('K_ring (off-d1):', np.diag(K_ring, 1))
print('K_ring [0,-1]  :', K_ring[0, -1])   # should be -k (ring closure)

## PINN hyper-parameters

In [ ]:
layers              = [1, 64, 64, N]
hyp_ini_weight_loss = np.array([100.0, 1.0])   # [beta_icx, beta_fx]
optimizer_LB_value  = True
lb = np.array([0.0])
ub = np.array([4.0])

## Initial conditions — broadband random perturbation

Small random displacements excite all spatial wavenumbers simultaneously,
so the 2-D FFT will reveal the full dispersion band.

In [ ]:
rng = np.random.default_rng(42)

x0_init  = rng.uniform(-0.05, 0.05, (1, N)).astype(np.float32)
xt0_init = np.zeros((1, N), dtype=np.float32)

y0_init  = np.zeros(N)           # impactors start flush with primary masses
yt0_init = np.full(N, -0.5)      # all impactors move downward initially

# Velocity-update matrix  A^{-1} B  (momentum + restitution)
A_mat   = np.array([[mx, my], [1.0, -1.0]])
B_mat   = np.array([[mx, my], [-r,   r  ]])
A_inv_B = np.linalg.inv(A_mat) @ B_mat

print('x0_init  :', np.round(x0_init, 4))
print('yt0_init :', yt0_init)

## Multi-segment PINN simulation

Each segment:
1. Train PINN on `[0, T_seg]` → frozen network  
2. Root-find impact times (Phase 2)  
3. Record `x_n(t)` up to the first impact  
4. Apply velocity update, propagate ICs to next segment  

Stop when accumulated time reaches `T_total`.

> **Note:** Each segment requires one full PINN training.  
> With `nIter=500` and L-BFGS polishing, expect ~2–5 min per segment.

In [ ]:
T_total   = 20.0    # total simulation time (s)  — increase for finer ω resolution
T_seg_max = 4.0     # maximum window per segment
num_pts   = 500     # collocation / output points per segment
nIter     = 500     # Adam iterations per segment

t0_arr = np.array([[0.0]])

# Running state
cur_x0   = x0_init.copy()
cur_xt0  = xt0_init.copy()
cur_y0   = y0_init.copy()
cur_yt0  = yt0_init.copy()
cur_phi  = np.array([[0.0]])
t_cum    = 0.0

# Storage
all_t_segs = []    # list of 1-D arrays (global time)
all_x_segs = []    # list of (num_pts+1, N) arrays

seg_idx = 0
while t_cum < T_total:
    seg_idx += 1
    T_win = min(T_seg_max, T_total - t_cum)
    if T_win < 1e-3:
        break

    print(f"\n{'='*60}")
    print(f"  Segment {seg_idx}  |  t_cum = {t_cum:.4f} s  |  window = {T_win:.4f} s")
    print(f"{'='*60}")

    t_arr = np.linspace(0.0, T_win, num_pts).reshape(-1, 1)

    model = PIPNNs(
        lb, ub,
        t0_arr, t_arr,
        cur_x0, cur_xt0,
        cur_y0, cur_yt0,
        M_mat, K_ring, D, N,
        cur_phi, phi1, phi2,
        layers, hyp_ini_weight_loss,
        C=C_mat,
        optimizer_LB=optimizer_LB_value,
    )
    model.train(nIter=nIter, optimizer_LB=optimizer_LB_value)

    t_impacts, hit = find_impact_times(model, cur_y0, cur_yt0, D, T_win)

    if hit.any():
        first_dof = int(np.argmin(t_impacts))
        t_end     = float(t_impacts[first_dof])
        print(f"  Impact: DOF {first_dof+1} at t = {t_end:.5f} s")
    else:
        t_end = T_win
        print(f"  No impact detected; using full window t_end = {t_end:.4f} s")

    # Predict on a fine uniform grid within this segment
    t_sim  = np.linspace(0.0, t_end, num_pts + 1).reshape(-1, 1)
    x_sim, xt_sim, _ = model.predict(t_sim)

    all_t_segs.append(t_sim.flatten() + t_cum)
    all_x_segs.append(x_sim)            # (num_pts+1, N)

    # Propagate ICs
    if hit.any():
        x_end_pred, xt_end_pred, _ = model.predict(np.array([[t_end]]))
        cur_x0, cur_xt0, cur_y0, cur_yt0 = propagate_ics(
            model, t_end,
            x_end_pred, xt_end_pred,
            cur_y0, cur_yt0,
            first_dof,
            mx * np.ones(N), my * np.ones(N), r,
            A_inv_B,
        )
    else:
        x_end_pred, xt_end_pred, _ = model.predict(np.array([[t_end]]))
        cur_x0  = x_end_pred
        cur_xt0 = xt_end_pred
        cur_y0  = cur_y0 + cur_yt0 * t_end
        # cur_yt0 unchanged

    t_cum   += t_end
    cur_phi  = cur_phi + t_end

print(f"\nSimulation complete: {seg_idx} segments, T_total = {t_cum:.4f} s")

## Stitch segments → uniform spatiotemporal array

Concatenate all segments and resample onto a **uniform** time grid required by the 2-D FFT.
Interpolation uses linear splines (sufficient for smooth PINN outputs).

In [ ]:
# Concatenate (duplicate boundary points at segment joints are fine for interpolation)
t_raw = np.concatenate(all_t_segs)           # (M_total,)
X_raw = np.vstack(all_x_segs).T              # (N, M_total)

# Resample to uniform grid
# Target dt chosen to resolve ω_max = 2*sqrt(k/mx)  with at least 20 points/cycle
omega_max_lin = 2.0 * np.sqrt(k / mx)        # rad/s — top of acoustic branch
dt_target     = (2.0 * np.pi / omega_max_lin) / 20.0
t_uniform     = np.arange(t_raw[0], t_raw[-1], dt_target)

X_uniform = np.zeros((N, len(t_uniform)))
for i in range(N):
    f_interp      = interp1d(t_raw, X_raw[i, :], kind='linear', fill_value='extrapolate')
    X_uniform[i, :] = f_interp(t_uniform)

print(f'Spatiotemporal array : {X_uniform.shape}   (N_dof × N_time)')
print(f'Time span            : {t_uniform[0]:.4f} → {t_uniform[-1]:.4f} s')
print(f'dt                   : {dt_target:.5f} s')
print(f'ω resolution         : {2*np.pi/(t_uniform[-1]-t_uniform[0]):.4f} rad/s')

## 2-D FFT → dispersion spectrum S(k, ω)

In [ ]:
k_pos, omega_pos, spectrum = dispersion_from_2dfft(
    t_uniform, X_uniform, skip_transient=0.15)

print(f'k range  : {k_pos[0]:.4f} → {k_pos[-1]:.4f} rad/unit-cell')
print(f'ω range  : {omega_pos[0]:.4f} → {omega_pos[-1]:.4f} rad/s')
print(f'Spectrum : {spectrum.shape}')

## Dispersion heatmap

Bright ridges are the nonlinear dispersion branches.  
The white dashed line is the **linear** acoustic dispersion  
$\omega_\mathrm{lin}(k) = 2\sqrt{K/M}\,|\sin(k/2)|$  (no impactors).

In [ ]:
fig, ax = plot_dispersion_heatmap(
    k_pos, omega_pos, spectrum,
    k_coupling=k, mx=mx,
    dB_range=40,
    save_path='pinn_dispersion_heatmap.png',
    figsize=(7, 5),
)

## Ridge extraction — nonlinear vs linear dispersion

In [ ]:
mpl.rcParams.update({
    'font.family':      'Times New Roman',
    'mathtext.fontset': 'custom',
    'mathtext.rm':      'Times New Roman',
    'mathtext.it':      'Times New Roman:italic',
    'pdf.fonttype':     42,
})
FS = 20
LW = 2.0

# Extract dominant ω per k (ignore DC component)
omega_min_ridge = 0.5   # rad/s
k_ridge, omega_ridge = extract_ridge(k_pos, omega_pos, spectrum,
                                     omega_min=omega_min_ridge)

# Linear reference
k_line  = np.linspace(0, np.pi, 300)
om_line = linear_dispersion(k_line, k, mx)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(k_line / np.pi, om_line, 'k--', lw=LW,
        label=r'Linear  ($D \to \infty$)')

valid = ~np.isnan(omega_ridge)
ax.plot(k_ridge[valid] / np.pi, omega_ridge[valid],
        'o-', color='C1', lw=LW, ms=7,
        label=f'PINN  ($D={D}$, $r={r}$)')

ax.set_xlabel(r'Wavenumber  $k/\pi$', fontsize=FS, labelpad=8)
ax.set_ylabel(r'Frequency  $\omega$  (rad/s)', fontsize=FS, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, prune='both'))
ax.tick_params(axis='both', labelsize=FS - 2)
ax.legend(fontsize=FS - 5, loc='upper left', framealpha=0.85)
ax.set_title('PINN dispersion curve vs linear baseline', fontsize=FS - 4)

plt.tight_layout(pad=1.5)
plt.savefig('pinn_dispersion_ridge.png', dpi=150, bbox_inches='tight')
plt.show()

# Print table
print(f"{'k/pi':>8}  {'omega_lin':>12}  {'omega_PINN':>12}  {'shift %':>9}")
k_discrete = k_pos[:N//2+1]
for kv, ov in zip(k_ridge[valid], omega_ridge[valid]):
    ol = linear_dispersion(kv, k, mx)
    shift = 100.0 * (ov - ol) / (ol + 1e-30)
    print(f"{kv/np.pi:>8.3f}  {ol:>12.4f}  {ov:>12.4f}  {shift:>+9.2f}")